In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The arithmetic of agent scale

**What you will learn.** You will learn how to go from "100,000 conversations a day" to tokens per minute,
turns in flight, Provisioned Throughput units and dollars. You will do it in the two minutes that a whiteboard
gives you. The full model is `scalelab/capacity.py` (about 120 lines). Read it one time, then use it.

> **The one-minute version.** A design review starts with envelope arithmetic. Say this sentence out loud:
> *"The unit of work is a turn. A turn is ~2.2 model calls of ~5k tokens. Thus 100k conversations a day is
> about 14 M input tokens a minute at peak. That is above the Flash tier baseline, and that is the
> constraint that I will design around, not Cloud Run."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 140)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The chain of multiplications

The chain goes from conversations/day to conversations/s, then to turns/s, then to model calls/s, then to
tokens/min. Every later number depends on this chain.

In [ ]:
from scalelab.capacity import Scenario, plan, plan_markdown, cost_per_call, gsus_needed, pt_usd_per_million_burndown, PRICES
s = Scenario()          # the anchor scenario: Meridian Mobile, 100k conversations/day
p = plan(s)
pd.DataFrame(p["rates"]).round(2)

In [ ]:
tokens = pd.DataFrame(p["tokens"]).T
tokens[["input_tpm", "uncached_input_tpm", "output_tpm"]].div(1e6).round(2).rename(columns=lambda c: c + " (M)").assign(vs_baseline=tokens.vs_baseline.round(2))

In [ ]:
ax = tokens.input_tpm.div(1e6).plot.bar(figsize=(6, 3.5), rot=0, title="Input tokens per minute vs the 10 M TPM Flash tier-3 baseline")
ax.axhline(s.tpm_baseline / 1e6, ls="--", c="grey"); ax.set_ylabel("M tokens / min"); plt.tight_layout()

The peak (×3) is already 1.4× the baseline. The incident (×10) is 4.6×. Nothing that you buy makes 46 M TPM
available in one minute. You must *shape* the incident instead (notebook 03/04). For the peak, you can buy
capacity (Provisioned Throughput) or decrease the load (fewer tokens per call).

## 2. Little's law: concurrency is latency × rate

$$
\text{Things in the system} = \text{arrival rate} \times \text{time each spends in it}.
$$

The turns in flight set the size of the orchestrator memory and of the model
concurrency. The concurrent *sessions* (users who think between turns) set the size of the open connections and of the hot state.

In [ ]:
conc = pd.DataFrame({k: v for k, v in p["concurrency"].items() if isinstance(v, dict)}).round(0)
print(conc)
print(f"\nthe 10 M TPM baseline sustains ≈ {p['concurrency']['max_turns_per_s_for_baseline']:.1f} turns/s "
      f"≈ {p['concurrency']['max_inflight_for_baseline']:.0f} turns in flight — the starting in-flight cap")

## 3. Cost per conversation

The bill counts cached input at 10 %. A Flash-Lite call costs a quarter of a Flash call. Route and cache *first*,
because every later percentage applies to the new baseline.

In [ ]:
cost = pd.Series(p["cost"]).round(4); print(cost)
ax = cost[["per_conversation_no_cache", "per_conversation_cached", "per_conversation_routed_cached"]].plot.bar(
    figsize=(6, 3.5), rot=0, title="$ per conversation"); ax.set_ylabel("USD"); plt.tight_layout()

## 4. Provisioned Throughput

One GSU of Gemini 3.5 Flash gives 675 *burndown* tokens/s (uncached input ×1, cached ×0.1, output ×6). PT is a
guarantee. It is not a discount, unless the term is long **and** the units stay busy.

In [ ]:
pt = p["pt"]
print("GSUs to carry the standard-model share:", pt["gsus"])
print("monthly cost of the average GSUs by term:", {k: round(v) for k, v in pt["monthly_usd_for_average_gsus"].items()})
pd.DataFrame({"$/M burndown": pt["usd_per_million_burndown"], "break-even utilisation": pt["breakeven_utilisation"]}).round(2)

## 5. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn

Fill in each function (replace the `todo()` call). Then run the check cell that comes after it. The practice notebook runs end to end when the exercises have no change. In the solutions notebook, every check passes.

#### (a) Little's law

Calculate the turns in flight and the concurrent sessions from rates and durations.

In [ ]:
def inflight_turns(turns_per_s, turn_seconds):
    return todo()

def concurrent_sessions(conversations_per_s, turns_per_conversation, turn_seconds, think_seconds):
    return todo()

In [ ]:
def _a():
    r = p["rates"]["peak"]
    assert abs(inflight_turns(r["turns_per_s"], s.turn_seconds) - 125) < 1
    assert abs(concurrent_sessions(r["conversations_per_s"], s.turns_per_conversation, s.turn_seconds, s.think_seconds) - 1375) < 5
    # Little's law holds for any rate and duration, not only the anchor scenario: L = λ·W
    for lam, w in [(0.5, 4.0), (3.0, 12.5), (40.0, 0.2), (7.3, 30.0)]:
        got = inflight_turns(lam, w)
        assert abs(got - lam * w) < 1e-6 * max(1.0, lam * w), f"inflight_turns({lam}, {w}) = {got}, expected λ·W = {lam * w:g}"
    for conv, turns, t, think in [(0.2, 5, 10.0, 30.0), (2.0, 1, 4.0, 0.0), (0.05, 12, 8.0, 60.0)]:
        want = conv * turns * (t + think)   # a session is in the system for its turns and the think time between them
        got = concurrent_sessions(conv, turns, t, think)
        assert abs(got - want) < 1e-6 * max(1.0, want), f"concurrent_sessions({conv}, {turns}, {t}, {think}) = {got}, expected {want:g}"
check("a: Little's law", _a)

#### (b) GSUs from burndown weights

From the price table, write a function that gives the same result as `gsus_needed`:

$$
\text{burndown} = \text{uncached} \times 1 + \text{cached} \times 0.1 + \text{output} \times \text{burn}.
$$

In [ ]:
def gsus(model, calls_per_s, input_tokens, output_tokens, cached_tokens):
    inp, out, cached_price, gsu_tokens_per_s, output_burn = PRICES[model]
    return todo()

In [ ]:
def _b():
    for calls in (1, 10, 29.8):
        assert abs(gsus("gemini-3.5-flash", calls, 5000, 350, 2700) - gsus_needed("gemini-3.5-flash", calls, 5000, 350, 2700)) < 1e-9
check("b: GSUs", _b)

#### (c) Break-even utilisation

Find the utilisation at which a GSU costs the same per burndown token as pay-as-you-go.

In [ ]:
def breakeven(model, term, input_tokens=5000, output_tokens=350, cached_tokens=2700):
    """PT $/M burndown at 100 % utilisation ÷ pay-as-you-go $/M burndown."""
    return todo()

In [ ]:
def _c():
    assert abs(breakeven("gemini-3.5-flash", "1y") - 0.75) < 0.02
    assert breakeven("gemini-3.5-flash", "1m") > 1.0   # a monthly term is never cheaper than pay-as-you-go
check("c: break-even", _c)

#### (d) Re-plan for a batch pipeline

The pipeline gets 2 M documents/day. It makes one call per document (3k in, 300 out, nothing cached). It processes the documents in an 8-hour window, with no think time. Build the Scenario. Then name the binding constraint.

In [ ]:
def batch_scenario():
    # hint: conversations_per_day = documents, turns 1, calls 1, incident_factor 1, peak_factor = 24/8
    return todo()

BINDING_CONSTRAINT = "..."   # one phrase

In [ ]:
def _d():
    b = plan(batch_scenario())
    assert b["tokens"]["peak"]["vs_baseline"] > 1.0, "the window should exceed the baseline"
    assert b["concurrency"]["peak"]["concurrent_sessions"] == b["concurrency"]["peak"]["inflight_turns"], "no think time"
    assert BINDING_CONSTRAINT != "..."
    print("   peak input TPM:", round(b["tokens"]["peak"]["input_tpm"] / 1e6, 1), "M; GSUs:", b["pt"]["gsus"]["peak"])
check("d: batch re-plan", _d)

## Takeaways

- Do the chain: conversations, turns, calls, tokens/min. Say the peak TPM and compare it with the tier baseline.
- Use Little's law two times: for in-flight turns (compute) and for concurrent sessions (connections, hot state).
- The cost levers change the cost by integer factors: route, cache, compact. On Flash, output is 6× the price of input.
- PT: say the break-even utilisation before anyone buys a monthly term.
- Cloud Run is ~0.1 % of the model bill. The token budget breaks first. Then the slowest downstream system breaks.

## Verify before relying on it

Examine these facts:

- The prices and model ids (5 Sep 2026 in `capacity.PRICES`).
- The PayGo tier baselines (Flash 2/4/10 M TPM).
- The PT GSU throughput and burndown weights.
- If cached tokens count against the PayGo baseline.